<a href="https://www.kaggle.com/code/gpreda/prohibition-orders-to-inhabit?scriptVersionId=259190600" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Data preparation

We should pay attention to the separator, this `csv` is using not the default separator but `;`.

In [2]:
df = pd.read_csv("/kaggle/input/arrts-dinterdiction-dhabiter/fr-219200631-arretes-d-interdiction-d-habiter.csv", sep=";")

In [3]:
df.head()

,objectid,Type de l'arrêté préfectoral,Numéro de voie,Références cadastrales,Nom de la voie,Type d'arrêté,geo_shape,geo_point_2d
0,20,Impropre à l'habitation,73,AS 54,Rue Adrien Cramail,Arrêté d'interdiction d'habiter valide,"{""coordinates"": [2.1696348121531335, 48.880280...","48.88028040943025, 2.1696348121531335"
1,18848,Insalubrité remédiable,7,AR 418,Rue du Quatre Septembre,Arrêté d'interdiction d'habiter valide,"{""coordinates"": [2.183992910731699, 48.8766778...","48.87667782206053, 2.183992910731699"
2,11,Insalubrité remédiable,21,AE 496,Rue Victor Troussard,Arrêté d'interdiction d'habiter NSP,"{""coordinates"": [2.181594033381362, 48.8830192...","48.88301927389812, 2.181594033381362"
3,26,Insalubrité remédiable,34,AH 183,Avenue Paul Doumer,Arrêté d'interdiction d'habiter NSP,"{""coordinates"": [2.1898786327238735, 48.883378...","48.8833787605052, 2.1898786327238735"
4,38,Impropre à l'habitation,217,BO 66,Avenue Napoléon Bonaparte,Arrêté d'interdiction d'habiter valide,"{""coordinates"": [2.16283289043061, 48.87223051...","48.87223051335277, 2.16283289043061"


We will translate for your convenience the column names:

- objectid → not changed
- Type de l'arrêté préfectoral → Type of prefectural order
- Numéro de voie → Street number
- Références cadastrales → Cadastral references (parcel reference / land registry code)
- Nom de la voie → Street name
- Type d'arrêté → Type of order
- geo_shape → not changed
- geo_point_2d → not changed

In [4]:
df.columns = ["objectid", "Type of prefectural order", "Street number", 
              "Cadastral references", "Street name", "Type of order",
             "geo_shape", "geo_point_2d"]

In [5]:
df.head()

,objectid,Type of prefectural order,Street number,Cadastral references,Street name,Type of order,geo_shape,geo_point_2d
0,20,Impropre à l'habitation,73,AS 54,Rue Adrien Cramail,Arrêté d'interdiction d'habiter valide,"{""coordinates"": [2.1696348121531335, 48.880280...","48.88028040943025, 2.1696348121531335"
1,18848,Insalubrité remédiable,7,AR 418,Rue du Quatre Septembre,Arrêté d'interdiction d'habiter valide,"{""coordinates"": [2.183992910731699, 48.8766778...","48.87667782206053, 2.183992910731699"
2,11,Insalubrité remédiable,21,AE 496,Rue Victor Troussard,Arrêté d'interdiction d'habiter NSP,"{""coordinates"": [2.181594033381362, 48.8830192...","48.88301927389812, 2.181594033381362"
3,26,Insalubrité remédiable,34,AH 183,Avenue Paul Doumer,Arrêté d'interdiction d'habiter NSP,"{""coordinates"": [2.1898786327238735, 48.883378...","48.8833787605052, 2.1898786327238735"
4,38,Impropre à l'habitation,217,BO 66,Avenue Napoléon Bonaparte,Arrêté d'interdiction d'habiter valide,"{""coordinates"": [2.16283289043061, 48.87223051...","48.87223051335277, 2.16283289043061"


In [6]:
df.shape

(39, 8)

# Data visualization

Let's prepare the lat/long values first.

In [7]:
def parse_latlon(s):
    lat, lon = [float(x.strip()) for x in str(s).split(",")]
    return lat, lon

df[["lat", "lon"]] = df["geo_point_2d"].apply(lambda s: pd.Series(parse_latlon(s)))

In [8]:

# Center map
m = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=14, control_scale=True)

# Colors by "Type d'arrêté"
type_colors = {
    "Arrêté d'interdiction d'habiter valide": "red",
    "Arrêté d'interdiction d'habiter NSP": "orange",
}
mc = MarkerCluster().add_to(m)

for _, r in df.iterrows():
    color = type_colors.get(r["Type of order"], "blue")
    popup = folium.Popup(
        f"<b>ObjectID:</b> {r['objectid']}<br>"
        f"<b>Type préfectoral:</b> {r['Type of prefectural order']}<br>"
        f"<b>Type d'arrêté:</b> {r['Type of order']}<br>"
        f"<b>Nom de la voie:</b> {r['Street name']}<br>"
        f"<b>Numéro de voie:</b> {r['Street number']}<br>"
        f"<b>Références cadastrales:</b> {r['Cadastral references']}",
        max_width=300
    )
    folium.Marker([r["lat"], r["lon"]],
                  tooltip=r["Street name"],
                  popup=popup,
                  icon=folium.Icon(color=color, icon="info-sign")).add_to(mc)



In [9]:
m